In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers
!pip install scikit-learn
!pip install tqdm
!pip install pandas

In [1]:
import csv
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tqdm import tqdm

# Data path and hyperparameters
RAW_DATA_DIR = "D:/comp-6713-industry-project/raw_data"
FINAL_MODEL_PATH = "./final_model"

BATCH_SIZE = 16
MAX_LENGTH = 256
EPOCHS = 30
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data loader
def load_csv_data(file_path):
    texts, labels = [], []
    valid_statuses = {'OnSite', 'Remote', 'Hybrid'}

    with open(file_path, 'r', newline='', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        for row_num, row in enumerate(reader, 1):
            if row_num == 1:
                continue  # Skip header
            if len(row) != 3:
                print(f"Row {row_num}: Invalid column count, skipped")
                continue
            job_id, text, status = row
            if status not in valid_statuses:
                print(f"Row {row_num}: Invalid status '{status}', skipped")
                continue
            texts.append(text.strip().replace('\n', ' '))
            labels.append(status.strip())
    return texts, labels

# Dataset
class JobDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer.encode_plus(
            text=str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Label encoder
def initialize_label_encoder():
    texts, labels = load_csv_data(f"{RAW_DATA_DIR}/work_arrangements_development_set.csv")
    encoder = LabelEncoder()
    encoder.fit(labels)
    return encoder

# Train the model
def train_model(label_encoder):
    train_texts, train_labels = load_csv_data(f"{RAW_DATA_DIR}/work_arrangements_development_set.csv")
    encoded_labels = label_encoder.transform(train_labels)

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=len(label_encoder.classes_)
    ).to(DEVICE)

    dataset = JobDataset(train_texts, encoded_labels, tokenizer, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for batch in progress_bar:
            optimizer.zero_grad()
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
            outputs = model(**inputs, labels=batch['labels'].to(DEVICE))
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        print(f"Epoch {epoch+1} Average Loss: {total_loss/len(dataloader):.4f}")

    model.save_pretrained(FINAL_MODEL_PATH)
    tokenizer.save_pretrained(FINAL_MODEL_PATH)
    print(f"\nModel saved to {FINAL_MODEL_PATH}")

# Evaluation metrics
def generate_test_report(model_path, label_encoder):
    test_texts, test_labels = load_csv_data(f"{RAW_DATA_DIR}/work_arrangements_test_set.csv")
    encoded_labels = label_encoder.transform(test_labels)

    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path).to(DEVICE)

    dataset = JobDataset(test_texts, encoded_labels, tokenizer, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)

    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            predictions.extend(preds)

    print("\nClassification Report:")
    print(classification_report(
        encoded_labels,
        predictions,
        target_names=label_encoder.classes_,
        digits=2
    ))

# Main function
if __name__ == "__main__":
    print("\nInitializing label encoder...")
    label_encoder = initialize_label_encoder()

    print("\nEvaluating pretrained model...")
    generate_test_report("bert-base-uncased", label_encoder)

    print("\nTraining new model...")
    train_model(label_encoder)

    print("\nEvaluating fine-tuned model...")
    generate_test_report(FINAL_MODEL_PATH, label_encoder)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Testing: 100%|███████████████████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  5.00it/s]
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,


Classification Report:
              precision    recall  f1-score   support

      Hybrid       0.29      0.89      0.44        27
      OnSite       0.71      0.26      0.38        46
      Remote       0.00      0.00      0.00        26

    accuracy                           0.36        99
   macro avg       0.33      0.38      0.27        99
weighted avg       0.41      0.36      0.30        99



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Song Yidong\AppData\Roaming\Python\Python312\site-packages\transformers\optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 1/30: 100%|████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  5.05it/s, loss=0.856]


Epoch 1 Average Loss: 1.0154


Epoch 2/30: 100%|████████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  4.90it/s, loss=0.984]


...

Epoch 29 Average Loss: 0.0191


Epoch 30/30: 100%|███████████████████████████████████████████████████████████| 7/7 [00:01<00:00,  4.47it/s, loss=0.035]


Epoch 30 Average Loss: 0.0199
Model saved to ./final_model


Testing: 100%|███████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00,  8.37it/s]


Classification Report:
              precision    recall  f1-score   support

      Hybrid       0.40      0.30      0.34        27
      OnSite       0.69      0.74      0.72        46
      Remote       0.53      0.62      0.57        26

    accuracy                           0.59        99
   macro avg       0.54      0.55      0.54        99
weighted avg       0.57      0.59      0.58        99

